# Phase 5 — Evaluation Report

**The most important notebook of the project.** This is where the core research question gets answered empirically.

Four models compared across three dimensions:
1. **Pass@1** (HumanEval) — functional correctness
2. **Mean cyclomatic complexity** — code simplicity
3. **Mean lint errors** — style adherence

The **star graph** is the ablation: DPO-execution-only vs. DPO-composite-reward. If the composite reward produces code with similar pass@1 but *lower* complexity and *fewer* lint errors, that is the empirical evidence that the composite reward avoids the spaghetti-code failure mode predicted in Phase 0.

This transforms the project from "I did SFT+DPO" to "I identified a known failure mode in the literature and demonstrated empirically that my solution works."

Dependencies (`pandas`, `matplotlib`, `seaborn`) are in the `dev` dependency group in `pyproject.toml`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.evaluation.evaluation_config import (
    BASE_METRICS, SFT_METRICS,
    DPO_COMPOSITE_METRICS, DPO_ABLATION_METRICS,
    STATIC_ANALYSIS_RESULTS,
)

def load_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None

# Load pass@1 metrics
metrics = {
    "Base": load_json(BASE_METRICS),
    "SFT": load_json(SFT_METRICS),
    "DPO (composite)": load_json(DPO_COMPOSITE_METRICS),
    "DPO (ablation)": load_json(DPO_ABLATION_METRICS),
}

# Load static analysis
try:
    static_df = pd.read_json(STATIC_ANALYSIS_RESULTS, lines=True)
    print(f"Static analysis: {len(static_df)} samples loaded")
except (FileNotFoundError, ValueError):
    static_df = pd.DataFrame()
    print("Static analysis not yet computed — run: uv run python -m src.evaluation.metrics_analyzer")

## 1. Pass@1 — Functional Correctness

The headline metric: what fraction of HumanEval problems does each model solve on the first attempt?

In [ ]:
rows = []
for name, m in metrics.items():
    if m:
        p1 = m.get("humaneval", {}).get("pass@1", 0) * 100
        rows.append({"Model": name, "pass@1 (%)": round(p1, 2)})
    else:
        rows.append({"Model": name, "pass@1 (%)": "pending"})

pass_df = pd.DataFrame(rows)
print(pass_df.to_string(index=False))

# Bar chart
plot_rows = [r for r in rows if isinstance(r["pass@1 (%)"], (int, float))]
if len(plot_rows) >= 2:
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = {"Base": "#9E9E9E", "SFT": "#2196F3",
              "DPO (composite)": "#4CAF50", "DPO (ablation)": "#FF9800"}
    names = [r["Model"] for r in plot_rows]
    scores = [r["pass@1 (%)"] for r in plot_rows]
    bar_colors = [colors.get(n, "#999") for n in names]
    bars = ax.bar(names, scores, color=bar_colors, width=0.5)
    ax.set_ylabel("HumanEval pass@1 (%)")
    ax.set_title("Functional Correctness: Base → SFT → DPO")
    ax.set_ylim(0, 100)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{score:.1f}%", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 2. Static Analysis — Code Quality Metrics

Pass@1 alone doesn't tell the whole story. A model could pass tests by producing needlessly complex code. These metrics measure *how* the code is written, not just *whether* it works.

In [ ]:
if len(static_df) > 0:
    # Aggregate per model
    model_order = ["base", "sft", "dpo_composite", "dpo_ablation"]
    labels = {"base": "Base", "sft": "SFT",
              "dpo_composite": "DPO (composite)", "dpo_ablation": "DPO (ablation)"}

    agg = static_df.groupby("model").agg(
        mean_cc=("complexity", "mean"),
        median_cc=("complexity", "median"),
        mean_lint=("lint_errors", "mean"),
        median_lint=("lint_errors", "median"),
    ).reindex([m for m in model_order if m in static_df["model"].unique()])

    agg.index = [labels.get(m, m) for m in agg.index]
    print(agg.round(2).to_string())
else:
    print("Static analysis not computed yet.")

## 3. ⭐ The Star Graph: Ablation — Composite vs. Execution-Only Reward

**This is what transforms the project from "I did SFT+DPO" to "I identified a real problem in the literature (reward hacking toward complex code), designed a solution (composite reward), and validated it empirically."**

The hypothesis: a binary execution-only reward ($R_{\text{exec}}$ alone) encourages spaghetti code — code that passes tests but is needlessly complex. The composite reward ($R_{\text{exec}} + R_{\text{static}} + R_{\text{style}}$) should produce code that is:
- **Equally correct** (similar pass@1)
- **Less complex** (lower cyclomatic complexity)
- **Cleaner** (fewer lint errors)

In [ ]:
if len(static_df) > 0 and all(m in static_df["model"].unique()
                               for m in ["dpo_composite", "dpo_ablation"]):
    comp = static_df[static_df["model"] == "dpo_composite"]
    abl = static_df[static_df["model"] == "dpo_ablation"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Cyclomatic Complexity ──
    cc_data = pd.DataFrame({
        "Composite Reward": comp["complexity"].values,
        "Execution-Only (ablation)": abl["complexity"].values,
    })
    cc_melted = cc_data.melt(var_name="Model", value_name="Cyclomatic Complexity")
    sns.boxplot(data=cc_melted, x="Model", y="Cyclomatic Complexity",
                palette={"Composite Reward": "#4CAF50",
                         "Execution-Only (ablation)": "#FF9800"}, ax=axes[0])
    axes[0].set_title("⭐ Cyclomatic Complexity: Composite vs. Ablation")

    # ── Lint Errors ──
    lint_data = pd.DataFrame({
        "Composite Reward": comp["lint_errors"].values,
        "Execution-Only (ablation)": abl["lint_errors"].values,
    })
    lint_melted = lint_data.melt(var_name="Model", value_name="Lint Errors")
    sns.boxplot(data=lint_melted, x="Model", y="Lint Errors",
                palette={"Composite Reward": "#4CAF50",
                         "Execution-Only (ablation)": "#FF9800"}, ax=axes[1])
    axes[1].set_title("Lint Errors: Composite vs. Ablation")

    plt.suptitle("Ablation Study: Does the Composite Reward Prevent Spaghetti Code?",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Summary table ──
    print(f"\n{'Metric':<25} {'Composite':>12} {'Ablation':>12} {'Δ':>8}")
    print("-" * 60)
    for metric, label in [("complexity", "Mean CC"), ("lint_errors", "Mean lint")]:
        c = comp[metric].mean()
        a = abl[metric].mean()
        print(f"{label:<25} {c:>12.2f} {a:>12.2f} {c - a:>+8.2f}")

    # pass@1 comparison
    comp_m = metrics.get("DPO (composite)")
    abl_m = metrics.get("DPO (ablation)")
    if comp_m and abl_m:
        c_p1 = comp_m["humaneval"]["pass@1"] * 100
        a_p1 = abl_m["humaneval"]["pass@1"] * 100
        print(f"{'Pass@1 (%)':25} {c_p1:>12.1f} {a_p1:>12.1f} {c_p1 - a_p1:>+8.1f}")
else:
    print("Both DPO models needed — run the ablation first (see README Phase 5).")

## 4. Four-Way Comparison: Full Pipeline Progression

Cyclomatic complexity and lint errors across all four models — shows how each training stage changes code quality.

In [ ]:
if len(static_df) > 0:
    model_order = ["base", "sft", "dpo_composite", "dpo_ablation"]
    labels = {"base": "Base", "sft": "SFT",
              "dpo_composite": "DPO\n(composite)", "dpo_ablation": "DPO\n(ablation)"}
    palette = {"Base": "#9E9E9E", "SFT": "#2196F3",
               "DPO\n(composite)": "#4CAF50", "DPO\n(ablation)": "#FF9800"}

    plot_df = static_df[static_df["model"].isin(model_order)].copy()
    plot_df["model"] = pd.Categorical(
        plot_df["model"].map(labels), categories=list(labels.values()), ordered=True
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.boxplot(data=plot_df, x="model", y="complexity", palette=palette, ax=axes[0])
    axes[0].set_title("Cyclomatic Complexity by Model")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Cyclomatic Complexity")

    sns.boxplot(data=plot_df, x="model", y="lint_errors", palette=palette, ax=axes[1])
    axes[1].set_title("Lint Errors by Model")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Lint Errors")

    plt.tight_layout()
    plt.show()

## 5. Qualitative Samples — Side by Side

5–10 representative examples where the composite model and the ablation model both solve the problem, but the composite model produces simpler code. These are generated by `src/evaluation/extract_qualitative.py` and saved to `data/evaluation/qualitative_samples.md`.

In [ ]:
from IPython.display import Markdown

qual_path = "data/evaluation/qualitative_samples.md"
try:
    with open(qual_path) as f:
        display(Markdown(f.read()))
except FileNotFoundError:
    print(f"Not generated yet — run: uv run python -m src.evaluation.extract_qualitative")

## 6. Summary Table

The complete picture — all metrics, all models, one table.

In [ ]:
if len(static_df) > 0:
    summary_rows = []
    model_configs = [
        ("Base", "base"),
        ("SFT", "sft"),
        ("DPO (composite)", "dpo_composite"),
        ("DPO (ablation)", "dpo_ablation"),
    ]
    for label, key in model_configs:
        m = metrics.get(label)
        p1 = m["humaneval"]["pass@1"] * 100 if m else None
        sub = static_df[static_df["model"] == key]
        summary_rows.append({
            "Model": label,
            "pass@1 (%)": f"{p1:.1f}" if p1 else "—",
            "Mean CC": f"{sub['complexity'].mean():.2f}" if len(sub) else "—",
            "Median CC": f"{sub['complexity'].median():.1f}" if len(sub) else "—",
            "Mean lint errors": f"{sub['lint_errors'].mean():.2f}" if len(sub) else "—",
        })
    print(pd.DataFrame(summary_rows).to_string(index=False))
else:
    print("Run evaluation first.")

## 7. Key Findings & Conclusions

*(Fill in after running Phase 5:)*

- **Pass@1 progression:** Base → SFT → DPO — [fill: did each stage improve?]
- **Composite vs. ablation — pass@1:** [fill: similar? composite didn't hurt correctness?]
- **Composite vs. ablation — CC:** [fill: composite has lower CC? by how much?]
- **Composite vs. ablation — lint:** [fill: composite has fewer lint errors? by how much?]
- **Hypothesis validated?** [fill: does the composite reward prevent spaghetti code?]
- **Qualitative evidence:** [fill: do the side-by-side samples support the quantitative findings?]

---

*This is the evidence that the project's core hypothesis — that a composite reward combining execution success with static-analysis metrics produces better-engineered code without sacrificing correctness — holds (or doesn't) in practice.*

Para terminar de pulir el notebook 05_evaluation_report.ipynb, ¿prefieres que en las gráficas de caja (boxplots) de complejidad ignoremos silenciosamente los valores NaN (código que falló el linter), o te gustaría que les asignáramos una penalización máxima explícita para que afecten negativamente la media de los modelos peores?

In [ ]:
if len(static_df) > 0 and all(m in static_df["model"].unique()
                               for m in ["dpo_composite", "dpo_ablation"]):
    # Extraer datos y filtrar los NaN (fallos sintácticos) para la comparativa
    comp = static_df[static_df["model"] == "dpo_composite"]
    abl = static_df[static_df["model"] == "dpo_ablation"]
    
    comp_valid = comp.dropna(subset=["complexity", "lint_errors"])
    abl_valid = abl.dropna(subset=["complexity", "lint_errors"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Cyclomatic Complexity ──
    cc_data = pd.DataFrame({
        "Composite Reward": comp_valid["complexity"].values,
        "Execution-Only (ablation)": abl_valid["complexity"].values,
    })
    cc_melted = cc_data.melt(var_name="Model", value_name="Cyclomatic Complexity")
    sns.boxplot(data=cc_melted, x="Model", y="Cyclomatic Complexity",
                palette={"Composite Reward": "#4CAF50",
                         "Execution-Only (ablation)": "#FF9800"}, ax=axes[0])
    axes[0].set_title("⭐ Cyclomatic Complexity: Composite vs. Ablation")

    # ── Lint Errors ──
    lint_data = pd.DataFrame({
        "Composite Reward": comp_valid["lint_errors"].values,
        "Execution-Only (ablation)": abl_valid["lint_errors"].values,
    })
    lint_melted = lint_data.melt(var_name="Model", value_name="Lint Errors")
    sns.boxplot(data=lint_melted, x="Model", y="Lint Errors",
                palette={"Composite Reward": "#4CAF50",
                         "Execution-Only (ablation)": "#FF9800"}, ax=axes[1])
    axes[1].set_title("Lint Errors: Composite vs. Ablation")

    plt.suptitle("Ablation Study: Does the Composite Reward Prevent Spaghetti Code?",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Summary table ──
    print(f"\n{'Metric':<25} {'Composite':>12} {'Ablation':>12} {'Δ':>8}")
    print("-" * 60)
    for metric, label in [("complexity", "Mean CC"), ("lint_errors", "Mean lint")]:
        c = comp_valid[metric].mean()
        a = abl_valid[metric].mean()
        print(f"{label:<25} {c:>12.2f} {a:>12.2f} {c - a:>+8.2f}")

    # pass@1 comparison
    comp_m = metrics.get("DPO (composite)")
    abl_m = metrics.get("DPO (ablation)")
    if comp_m and abl_m:
        c_p1 = comp_m["humaneval"]["pass@1"] * 100
        a_p1 = abl_m["humaneval"]["pass@1"] * 100
        print(f"{'Pass@1 (%)':25} {c_p1:>12.1f} {a_p1:>12.1f} {c_p1 - a_p1:>+8.1f}")
else:
    print("Both DPO models needed — run the ablation first (see README Phase 5).")

In [ ]:
if len(static_df) > 0:
    summary_rows = []
    model_configs = [
        ("Base", "base"),
        ("SFT", "sft"),
        ("DPO (composite)", "dpo_composite"),
        ("DPO (ablation)", "dpo_ablation"),
    ]
    for label, key in model_configs:
        m = metrics.get(label)
        p1 = m["humaneval"]["pass@1"] * 100 if m else None
        
        sub = static_df[static_df["model"] == key]
        # Calcular tasa de fallos sintácticos (NaNs en la métrica de complejidad)
        parse_failure_rate = sub['complexity'].isna().mean() * 100 if len(sub) > 0 else 0
        
        # Filtrar el código válido para extraer medias precisas de calidad
        sub_valid = sub.dropna(subset=["complexity", "lint_errors"])
        
        summary_rows.append({
            "Model": label,
            "pass@1 (%)": f"{p1:.1f}" if p1 else "—",
            "Parse Failure (%)": f"{parse_failure_rate:.1f}",
            "Mean CC": f"{sub_valid['complexity'].mean():.2f}" if len(sub_valid) else "—",
            "Median CC": f"{sub_valid['complexity'].median():.1f}" if len(sub_valid) else "—",
            "Mean lint errors": f"{sub_valid['lint_errors'].mean():.2f}" if len(sub_valid) else "—",
        })
    print(pd.DataFrame(summary_rows).to_string(index=False))
else:
    print("Run evaluation first.")